<img src="../../branding/logo.png" alt="Ragdoll" width="120" />

# Ragdoll - Data Ingestion Tutorial

This notebook walks through the different ways to ingest content into Ragdoll
and how to retrieve it again with RAG:

1. **Plain text** - content is sent directly to the API as a string.
2. **URL source** - a web page (`https://example.com/`) is registered and fetched by the worker.
3. **File upload** - the documents in `test_documents/` are uploaded base64-encoded (`.md`, `.csv`, `.json`, `.txt`).

**Structure of every ingestion cell**
- The source is pushed into the queue; the `source_id` and `job_id` are taken from the response.
- The status is then polled **every 2 seconds** (`pending` -> `processing` -> `completed`).
- Every ingestion sets meaningful **metadata**.

**After every ingestion cell** there is a retrieval cell that filters on the
same metadata and asks one or two fitting example questions.

> **Prerequisites**
> - A running Ragdoll server at `http://127.0.0.1:8080` (e.g. via
>   `./scripts/dev-local.sh` from the repo root) — see
>   [../getting-started.md](../getting-started.md).
> - The embedding/rerank models finished downloading
>   (`/api/v1/health` returns `"ready": true`).
> - Start this notebook from the `docs/tutorial/` directory so the relative path
>   `test_documents/` resolves correctly.

## The test documents and their overlaps

All documents belong to the fictional *Lumen Analytics GmbH*. Each document
deliberately covers **several topics**, and some topics **overlap** between
documents - ideal for testing vector search and metadata filters:

| Topic | Appears in |
|---|---|
| Remote / mobile work | Employee handbook (HR view) + IT security policy (VPN view) |
| Pricing / plans | Product FAQ (prose) + Pricing plans (`.json`, structured) |
| Onboarding | Employee handbook (section) + Onboarding checklist (`.txt`, step by step) |
| Berlin office / founded 2016 | Inline company profile + Office locations (`.csv`) |

At the end of the notebook, a section **without filters** demonstrates how a
single question returns hits from multiple overlapping documents.

In [1]:
import base64
import json
import time
from pathlib import Path

import requests
from requests.exceptions import ConnectionError as RequestsConnectionError

BASE_URL = "http://127.0.0.1:8080"
API_URL = f"{BASE_URL}/api/v1"
RELEASE = "first-release"
DOC_DIR = Path("test_documents")
POLL_INTERVAL = 2.0  # seconds between status checks


def wait_for_server(base_url: str = BASE_URL, timeout: float = 120.0, interval: float = 2.0) -> None:
    """Wait until the Ragdoll gateway responds on /api/v1/health."""
    deadline = time.time() + timeout
    print(f"Waiting for Ragdoll at {base_url} ...")
    while time.time() < deadline:
        try:
            resp = requests.get(f"{base_url}/api/v1/health", timeout=5)
            if resp.ok and resp.json().get("ready"):
                print("Server is ready.\n")
                return
        except RequestsConnectionError:
            pass
        time.sleep(interval)
    raise RuntimeError(
        f"Ragdoll is not reachable at {base_url}.\n"
        "Start it from the ragdoll/ directory:\n"
        "  ./scripts/dev-local.sh start"
    )


wait_for_server()

# --- Login: obtain a session token ---
resp = requests.post(
    f"{API_URL}/auth/login",
    json={"email": "admin@ragdoll.ai", "password": "admin"},
    timeout=30,
)
resp.raise_for_status()
TOKEN = resp.json()["token"]
HEADERS = {"Authorization": f"Bearer {TOKEN}"}
print("Login ok - token received.")


def enqueue(source: dict) -> dict:
    """Push exactly one source into the ingestion queue.

    Returns the batch response `result` object:
    {"source_id": ..., "job_id": ...}.
    """
    resp = requests.post(
        f"{API_URL}/releases/{RELEASE}/sources",
        headers=HEADERS,
        json=[source],  # the API always expects an array of sources
        timeout=60,
    )
    resp.raise_for_status()
    item = resp.json()["items"][0]
    if item.get("error"):
        raise RuntimeError(f"Enqueue failed: {item['error']}")
    return item["result"]


def poll_job(source_id: str, job_id: str, interval: float = POLL_INTERVAL) -> dict:
    """Poll the source status every `interval` seconds until completed/failed."""
    flt = json.dumps({"field": "id", "op": "eq", "value": source_id})
    print(f"Job {job_id}")
    print(f"  -> source {source_id} queued, polling every {interval:.0f}s ...")
    last_status = None
    waited = 0.0
    while True:
        resp = requests.get(
            f"{API_URL}/releases/{RELEASE}/sources",
            headers=HEADERS,
            params={"filter": flt},
            timeout=30,
        )
        resp.raise_for_status()
        rows = resp.json()
        record = rows[0] if rows else None
        status = record["status"] if record else "unknown"
        if status != last_status:
            print(f"  [t+{waited:>4.0f}s] status: {status}")
            last_status = status
        if status == "completed":
            print("  Ingestion done.\n")
            return record
        if status == "failed":
            print(f"  ERROR: {record.get('error')}\n")
            return record
        time.sleep(interval)
        waited += interval


def ingest(source: dict) -> dict:
    """Enqueue + polling in one step."""
    job = enqueue(source)
    return poll_job(job["source_id"], job["job_id"])


def ingest_file(path: Path, metadata: dict, name: str | None = None) -> dict:
    """Read a file, base64-encode it, and ingest it as a file source."""
    data = path.read_bytes()
    source = {
        "type": "file",
        "name": name or path.name,
        "content": base64.b64encode(data).decode("ascii"),
        "metadata": metadata,
    }
    print(f"File: {path}  ({len(data)} bytes)")
    return ingest(source)


def _api_error(resp: requests.Response, action: str) -> RuntimeError:
    """Build a readable error from a Ragdoll batch or problem-details response."""
    try:
        body = resp.json()
    except ValueError:
        body = resp.text
    if isinstance(body, dict) and "items" in body:
        item = body["items"][0]
        if item.get("error"):
            err = item["error"]
            detail = err.get("detail") if isinstance(err, dict) else err
            return RuntimeError(f"{action} failed ({resp.status_code}): {detail}")
    if isinstance(body, dict) and body.get("detail"):
        return RuntimeError(f"{action} failed ({resp.status_code}): {body['detail']}")
    return RuntimeError(f"{action} failed ({resp.status_code}): {body}")


def retrieve(question: str, flt: dict | None = None, top_k: int = 3, rerank: bool = True) -> list[dict]:
    """Ask a question against the RAG index, optionally with a metadata filter."""
    req = {"text": question, "top_k": top_k, "rerank": rerank}
    if flt is not None:
        req["filter"] = flt
    resp = requests.post(
        f"{API_URL}/releases/{RELEASE}/queries",
        headers=HEADERS,
        params={"store_payload": "true"},
        json=[req],
        timeout=120,
    )
    if not resp.ok:
        raise _api_error(resp, "Query")
    item = resp.json()["items"][0]
    if item.get("error"):
        raise RuntimeError(f"Query failed: {item['error']}")
    matches = item["result"]["matches"]
    print(f"Question: {question}")
    if flt is not None:
        print(f"Filter: {json.dumps(flt, ensure_ascii=False)}")
    if not matches:
        print("  (no matches)")
    for i, m in enumerate(matches, 1):
        score = m.get("rerank_score")
        score = score if score is not None else m.get("semantic_score")
        snippet = " ".join(m["content"].split())[:160]
        print(
            f"  {i}. score={score:.3f} | source={m['source_name']} "
            f"| meta={json.dumps(m['metadata'], ensure_ascii=False)}"
        )
        print(f"     {snippet}...")
    print()
    return matches


def suggest(questions: list[str], flt: dict | None = None, top_k: int = 3) -> None:
    """Ask one or two fitting example questions and show the matches."""
    for q in questions:
        retrieve(q, flt=flt, top_k=top_k)


print("Helpers ready: ingest(), ingest_file(), retrieve(), suggest()")

Waiting for Ragdoll at http://127.0.0.1:8080 ...
Server is ready.

Login ok - token received.
Helpers ready: ingest(), ingest_file(), retrieve(), suggest()


## 0. Smallest possible example

Before the structured walkthrough, here is the entire loop in two lines: ingest
one piece of text, then ask a question about it. Everything after this is just
variations on these two calls.

In [ ]:
ingest({
    "type": "text",
    "name": "smoke-test",
    "content": "Ragdoll is a fully local, retrieval-only RAG pipeline.",
})

retrieve("What is Ragdoll?")

## 1. Plain text upload

The simplest path: we send text directly as `content` (`type: "text"`). The
content is a short company profile - it deliberately overlaps with
`office_locations.csv` (Berlin, founded 2016) and the product FAQ (Lumen
Insights Platform).

In [2]:
text_source = {
    "type": "text",
    "name": "lumen-company-profile",
    "content": (
        "Lumen Analytics GmbH was founded in 2016 in Berlin and operates the "
        "SaaS solution Lumen Insights Platform. The company employs around 220 "
        "people across several European locations and helps customers turn raw "
        "data into understandable metrics and dashboards."
    ),
    "metadata": {
        "category": "company",
        "doc_type": "about",
        "format": "text",
        "language": "en",
        "source": "inline",
    },
}

record = ingest(text_source)

Job a971e358-7c62-44be-81a5-6bdcb4b63389
  -> source abc82a4d-70ca-4065-b2d1-251f11b9c020 queued, polling every 2s ...
  [t+   0s] status: pending
  [t+   2s] status: completed
  Ingestion done.



In [3]:
# Retrieval: same metadata filter (category=company), 1-2 fitting questions
suggest(
    [
        "When and where was Lumen Analytics founded?",
        "What does the Lumen Insights Platform do?",
    ],
    flt={"field": "meta.category", "op": "eq", "value": "company"},
)

Question: When and where was Lumen Analytics founded?
Filter: {"field": "meta.category", "op": "eq", "value": "company"}
  1. score=1.000 | source=lumen-company-profile | meta={"category": "company", "doc_type": "about", "format": "text", "language": "en", "source": "inline", "unit_kinds": ["paragraph"]}
     Lumen Analytics GmbH was founded in 2016 in Berlin and operates the SaaS solution Lumen Insights Platform. The company employs around 220 people across several ...

Question: What does the Lumen Insights Platform do?
Filter: {"field": "meta.category", "op": "eq", "value": "company"}
  1. score=1.000 | source=lumen-company-profile | meta={"category": "company", "doc_type": "about", "format": "text", "language": "en", "source": "inline", "unit_kinds": ["paragraph"]}
     Lumen Analytics GmbH was founded in 2016 in Berlin and operates the SaaS solution Lumen Insights Platform. The company employs around 220 people across several ...



## 2. URL source

Instead of text we only register a URL (`type: "url"`). The worker fetches the
page, extracts the readable text, and chunks it. We use `https://example.com/`.

In [4]:
url_source = {
    "type": "url",
    "name": "example-domain",
    "url": "https://example.com/",
    "metadata": {
        "category": "reference",
        "doc_type": "webpage",
        "format": "url",
        "language": "en",
        "source": "example.com",
    },
}

record = ingest(url_source)

Job 8c23af77-b651-4c9d-a1df-0ec3c3129d22
  -> source 8cc7b3c8-52b0-4bd0-b54d-bce135f3bb71 queued, polling every 2s ...
  [t+   0s] status: pending
  [t+   2s] status: completed
  Ingestion done.



In [5]:
# Retrieval: filter on source=example.com
suggest(
    [
        "What is the example domain used for?",
        "Can example.com be used in documents without prior permission?",
    ],
    flt={"field": "meta.source", "op": "eq", "value": "example.com"},
)

Question: What is the example domain used for?
Filter: {"field": "meta.source", "op": "eq", "value": "example.com"}
  1. score=1.000 | source=example-domain | meta={"category": "reference", "doc_type": "webpage", "format": "url", "language": "en", "source": "example.com", "unit_kinds": ["paragraph", "paragraph"]}
     This domain is for use in documentation examples without needing permission. Avoid use in operations. Learn more...

Question: Can example.com be used in documents without prior permission?
Filter: {"field": "meta.source", "op": "eq", "value": "example.com"}
  1. score=1.000 | source=example-domain | meta={"category": "reference", "doc_type": "webpage", "format": "url", "language": "en", "source": "example.com", "unit_kinds": ["paragraph", "paragraph"]}
     This domain is for use in documentation examples without needing permission. Avoid use in operations. Learn more...



## 3. File uploads

From here on, each cell uploads exactly **one** document from `test_documents/`
as a file (`type: "file"`, base64-encoded content). Each file gets a unique
`doc_type`, so the following retrieval cell can filter precisely on that one
document.

### 3.1 Employee handbook (`.md`)

In [6]:
record = ingest_file(
    DOC_DIR / "employee_handbook.md",
    metadata={
        "category": "hr",
        "doc_type": "handbook",
        "format": "md",
        "language": "en",
        "team": "people",
        "confidentiality": "internal",
    },
)

File: test_documents/employee_handbook.md  (4793 bytes)
Job 3fd3149b-2d51-4350-a6d5-81a5ac6e414e
  -> source fa3de6a1-9166-470a-b4fc-4449ba0fe3e2 queued, polling every 2s ...
  [t+   0s] status: pending
  [t+  14s] status: completed
  Ingestion done.



In [7]:
# Retrieval: filter on doc_type=handbook
suggest(
    [
        "How many days of remote work are allowed per week?",
        "How many vacation days do I get per year?",
    ],
    flt={"field": "meta.doc_type", "op": "eq", "value": "handbook"},
)

Question: How many days of remote work are allowed per week?
Filter: {"field": "meta.doc_type", "op": "eq", "value": "handbook"}
  1. score=1.000 | source=employee_handbook.md | meta={"category": "hr", "confidentiality": "internal", "doc_type": "handbook", "format": "md", "language": "en", "section_path": ["Employee Handbook - Lumen Analytics GmbH", "Working Hours and Flextime"], "team": "people", "unit_kinds": ["paragraph", "heading", "paragraph", "paragraph", "heading", "paragraph"]}
     [Employee Handbook - Lumen Analytics GmbH > Working Hours and Flextime] Overtime is recorded on a working-time account and can be compensated as time off rather...
  2. score=0.517 | source=employee_handbook.md | meta={"category": "hr", "confidentiality": "internal", "doc_type": "handbook", "format": "md", "language": "en", "section_path": ["Employee Handbook - Lumen Analytics GmbH"], "team": "people", "unit_kinds": ["heading", "paragraph", "heading", "paragraph"]}
     [Employee Handbook - Lumen An

### 3.2 IT security policy (`.md`)

In [8]:
record = ingest_file(
    DOC_DIR / "it_security_policy.md",
    metadata={
        "category": "security",
        "doc_type": "policy",
        "format": "md",
        "language": "en",
        "team": "it",
        "confidentiality": "confidential",
    },
)

File: test_documents/it_security_policy.md  (4536 bytes)
Job 681f1ac7-2d19-40b7-baae-213bd5093ccf
  -> source 85851d0d-9380-4fcc-97dc-4139251454aa queued, polling every 2s ...
  [t+   0s] status: pending
  [t+  12s] status: completed
  Ingestion done.



In [9]:
# Retrieval: filter on doc_type=policy
suggest(
    [
        "What is the minimum password length?",
        "When is using the VPN mandatory?",
    ],
    flt={"field": "meta.doc_type", "op": "eq", "value": "policy"},
)

Question: What is the minimum password length?
Filter: {"field": "meta.doc_type", "op": "eq", "value": "policy"}
  1. score=1.000 | source=it_security_policy.md | meta={"category": "security", "confidentiality": "confidential", "doc_type": "policy", "format": "md", "language": "en", "section_path": ["IT Security Policy - Lumen Analytics GmbH"], "team": "it", "unit_kinds": ["heading", "paragraph", "heading", "paragraph", "paragraph", "heading"]}
     [IT Security Policy - Lumen Analytics GmbH] # IT Security Policy - Lumen Analytics GmbH This policy is binding for all employees and describes the technical and...
  2. score=0.291 | source=it_security_policy.md | meta={"category": "security", "confidentiality": "confidential", "doc_type": "policy", "format": "md", "language": "en", "section_path": ["IT Security Policy - Lumen Analytics GmbH", "Data Classification"], "team": "it", "unit_kinds": ["paragraph", "paragraph", "heading", "paragraph", "paragraph", "heading"]}
     [IT Security Pol

### 3.3 Product FAQ (`.md`)

In [10]:
record = ingest_file(
    DOC_DIR / "product_faq.md",
    metadata={
        "category": "product",
        "doc_type": "faq",
        "format": "md",
        "language": "en",
        "audience": "customer",
        "confidentiality": "public",
    },
)

File: test_documents/product_faq.md  (3712 bytes)
Job a386354a-a793-456e-8aa2-b863e93ed24e
  -> source af8562df-1846-49a4-8e3a-ab1816ca0973 queued, polling every 2s ...
  [t+   0s] status: pending
  [t+   8s] status: completed
  Ingestion done.



In [11]:
# Retrieval: filter on doc_type=faq
suggest(
    [
        "What pricing plans are available?",
        "How long is my data retained?",
    ],
    flt={"field": "meta.doc_type", "op": "eq", "value": "faq"},
)

Question: What pricing plans are available?
Filter: {"field": "meta.doc_type", "op": "eq", "value": "faq"}
  1. score=1.000 | source=product_faq.md | meta={"audience": "customer", "category": "product", "confidentiality": "public", "doc_type": "faq", "format": "md", "language": "en", "section_path": ["Product FAQ - Lumen Insights Platform"], "unit_kinds": ["heading", "paragraph", "heading", "paragraph"]}
     [Product FAQ - Lumen Insights Platform] # Product FAQ - Lumen Insights Platform Frequently asked questions from our customers about the Lumen Insights Platform,...
  2. score=0.740 | source=product_faq.md | meta={"audience": "customer", "category": "product", "confidentiality": "public", "doc_type": "faq", "format": "md", "language": "en", "section_path": ["Product FAQ - Lumen Insights Platform", "What pricing plans are available?"], "unit_kinds": ["paragraph", "heading", "paragraph", "paragraph", "heading", "paragraph", "paragraph", "heading"]}
     [Product FAQ - Lumen Insights 

### 3.4 Office locations (`.csv`)

In [12]:
record = ingest_file(
    DOC_DIR / "office_locations.csv",
    metadata={
        "category": "facilities",
        "doc_type": "locations",
        "format": "csv",
        "language": "en",
        "team": "ops",
        "confidentiality": "internal",
    },
)

File: test_documents/office_locations.csv  (664 bytes)
Job 833804da-86e1-4e7c-b371-6593edbef3b7
  -> source c0b45971-ef27-44fb-8d61-3eee51be9163 queued, polling every 2s ...
  [t+   0s] status: pending
  [t+   2s] status: completed
  Ingestion done.



In [13]:
# Retrieval: filter on doc_type=locations
suggest(
    [
        "Which office is the headquarters of Lumen Analytics?",
        "How many employees work at the Lisbon office?",
    ],
    flt={"field": "meta.doc_type", "op": "eq", "value": "locations"},
)

Question: Which office is the headquarters of Lumen Analytics?
Filter: {"field": "meta.doc_type", "op": "eq", "value": "locations"}
  1. score=1.000 | source=office_locations.csv | meta={"category": "facilities", "confidentiality": "internal", "doc_type": "locations", "format": "csv", "language": "en", "team": "ops", "unit_kinds": ["paragraph"]}
     city,country,opened,employees,focus,timezone,remote_friendly Berlin,Germany,2016,84,Product development and headquarters,Europe/Berlin,yes Munich,Germany,2019,3...

Question: How many employees work at the Lisbon office?
Filter: {"field": "meta.doc_type", "op": "eq", "value": "locations"}
  1. score=1.000 | source=office_locations.csv | meta={"category": "facilities", "confidentiality": "internal", "doc_type": "locations", "format": "csv", "language": "en", "team": "ops", "unit_kinds": ["paragraph"]}
     city,country,opened,employees,focus,timezone,remote_friendly Berlin,Germany,2016,84,Product development and headquarters,Europe/Berlin,y

### 3.5 Pricing plans (`.json`)

In [14]:
record = ingest_file(
    DOC_DIR / "pricing_plans.json",
    metadata={
        "category": "product",
        "doc_type": "pricing",
        "format": "json",
        "language": "en",
        "audience": "customer",
        "confidentiality": "public",
    },
)

File: test_documents/pricing_plans.json  (1740 bytes)
Job 11cfe60e-4ca9-4bd9-ab1c-5681ffdc9704
  -> source efb39d17-5840-4020-8684-7a78c2ef87f5 queued, polling every 2s ...
  [t+   0s] status: pending
  [t+   6s] status: completed
  Ingestion done.



In [15]:
# Retrieval: filter on doc_type=pricing
suggest(
    [
        "How much does the Team plan cost per month?",
        "Which plan offers single sign-on?",
    ],
    flt={"field": "meta.doc_type", "op": "eq", "value": "pricing"},
)

Question: How much does the Team plan cost per month?
Filter: {"field": "meta.doc_type", "op": "eq", "value": "pricing"}
  1. score=1.000 | source=pricing_plans.json | meta={"audience": "customer", "category": "product", "confidentiality": "public", "doc_type": "pricing", "format": "json", "language": "en", "unit_kinds": ["paragraph", "paragraph", "paragraph", "paragraph", "paragraph", "paragraph", "paragraph", "paragraph", "paragraph", "paragraph", "paragraph", "paragraph"]}
     { "name": "Team", "price_per_month": 199, "data_retention_days": 180, "included_users": 15, "api_rate_limit_per_minute": 600, "support": "Email and chat", "feat...
  2. score=0.465 | source=pricing_plans.json | meta={"audience": "customer", "category": "product", "confidentiality": "public", "doc_type": "pricing", "format": "json", "language": "en", "unit_kinds": ["paragraph", "paragraph", "paragraph", "paragraph", "paragraph", "paragraph", "paragraph", "paragraph", "paragraph", "paragraph", "paragraph", "par

### 3.6 Onboarding checklist (`.txt`)

In [16]:
record = ingest_file(
    DOC_DIR / "onboarding_checklist.txt",
    metadata={
        "category": "hr",
        "doc_type": "checklist",
        "format": "txt",
        "language": "en",
        "team": "people",
        "confidentiality": "internal",
    },
)

File: test_documents/onboarding_checklist.txt  (2305 bytes)
Job 76e50625-4e5e-4cec-9dbb-5619f5e54e2a
  -> source 0628581b-a432-4134-8d32-1723942bc0da queued, polling every 2s ...
  [t+   0s] status: pending
  [t+   2s] status: completed
  Ingestion done.



In [17]:
# Retrieval: filter on doc_type=checklist
suggest(
    [
        "What needs to be done on the first day of onboarding?",
        "What happens after 30 days of onboarding?",
    ],
    flt={"field": "meta.doc_type", "op": "eq", "value": "checklist"},
)

Question: What needs to be done on the first day of onboarding?
Filter: {"field": "meta.doc_type", "op": "eq", "value": "checklist"}
  1. score=1.000 | source=onboarding_checklist.txt | meta={"category": "hr", "confidentiality": "internal", "doc_type": "checklist", "format": "txt", "language": "en", "team": "people", "unit_kinds": ["paragraph", "paragraph", "paragraph", "list_item", "list_item", "list_item", "list_item", "list_item", "list_item", "list_item", "paragraph", "list_item", "list_item", "list_item", "list_item", "list_item", "list_item", "list_item", "paragraph", "list_item", "list_item", "list_item", "list_item", "list_item", "list_item", "paragraph", "list_item", "list_item", "list_item", "list_item", "list_item", "paragraph", "list_item", "list_item", "paragraph", "list_item"]}
     Onboarding Checklist for New Employees - Lumen Analytics GmbH This checklist guides the first 30 days and complements the employee handbook. It is internal and ...

Question: What happens afte

## 4. Overlap & vector search (without filters)

Now we query **without a metadata filter** across the whole index. This shows
how vector search returns the relevant chunks for a topic from **multiple**
documents - exactly where the contents overlap. Watch the `source` column in
the output: with good overlap, different documents show up together.

In [18]:
# Remote work -> employee handbook (HR) AND IT security policy (VPN)
retrieve("What are the rules for working from home?", top_k=5)

# Pricing -> product FAQ (prose) AND pricing plans (.json)
retrieve("How much does it cost to use the platform?", top_k=5)

# Onboarding -> employee handbook AND onboarding checklist
retrieve("What is part of onboarding new employees?", top_k=5)

Question: What are the rules for working from home?
  1. score=1.000 | source=employee_handbook.md | meta={"category": "hr", "confidentiality": "internal", "doc_type": "handbook", "format": "md", "language": "en", "section_path": ["Employee Handbook - Lumen Analytics GmbH", "Working Hours and Flextime"], "team": "people", "unit_kinds": ["paragraph", "heading", "paragraph", "paragraph", "heading", "paragraph"]}
     [Employee Handbook - Lumen Analytics GmbH > Working Hours and Flextime] Overtime is recorded on a working-time account and can be compensated as time off rather...
  2. score=0.851 | source=it_security_policy.md | meta={"category": "security", "confidentiality": "confidential", "doc_type": "policy", "format": "md", "language": "en", "section_path": ["IT Security Policy - Lumen Analytics GmbH", "Data Classification"], "team": "it", "unit_kinds": ["paragraph", "paragraph", "heading", "paragraph", "paragraph", "heading"]}
     [IT Security Policy - Lumen Analytics GmbH > Data C

[{'chunk_id': 'cbb7f206-90c9-41cf-a8d9-7feb920ae040',
  'source_id': 'fa3de6a1-9166-470a-b4fc-4449ba0fe3e2',
  'source_name': 'employee_handbook.md',
  'content': '[Employee Handbook - Lumen Analytics GmbH > Onboarding New Colleagues]\n\nNew employees go through a structured onboarding during their first four weeks.\nOn the first day, the laptop, accounts, and the mentoring program are set up. In\nthe first week there are introductions to the product, the processes, and the\nmandatory security training. The detailed task list can be found in the separate\nOnboarding Checklist. After 30 days there is a first feedback conversation with\nthe team lead.',
  'metadata': {'category': 'hr',
   'confidentiality': 'internal',
   'doc_type': 'handbook',
   'format': 'md',
   'language': 'en',
   'section_path': ['Employee Handbook - Lumen Analytics GmbH',
    'Onboarding New Colleagues'],
   'team': 'people',
   'unit_kinds': ['paragraph']},
  'provenance': [{'end': 3689, 'start': 3274}],
  'sem

## Next steps

You have now ingested every source type and seen filtered and unfiltered
retrieval. To go further:

- **Filtering & rerank** — the full filter DSL and query options:
  [../querying.md](../querying.md)
- **Chunk quality** — tune Semantic Split for your documents:
  [../chunking.md](../chunking.md)
- **Releases & stages** — version content and serve production traffic:
  [../concepts.md](../concepts.md)
- **Common traps** — the breaking points worth knowing:
  [../pitfalls.md](../pitfalls.md)